# 07 - Deployment Artifacts

## CyberForge AI - Final Deployment Package

This notebook creates final deployment artifacts for:
- Hugging Face Hub upload
- Production deployment
- Model versioning and documentation

### Outputs:
- Complete model package for HF Hub
- Docker configuration for ML services
- Environment configuration
- Deployment documentation

In [ ]:
import json
import os
import time
import shutil
from pathlib import Path
from typing import Dict, List, Any
import warnings
warnings.filterwarnings('ignore')

# Configuration
config_path = Path("notebook_config.json")
if not config_path.exists():
    config_path = Path("/home/user/app/notebooks/notebook_config.json")
with open(config_path) as f:
    CONFIG = json.load(f)

MODELS_DIR = Path(CONFIG["datasets_dir"]).parent / "models"
BACKEND_DIR = MODELS_DIR.parent / "backend_package"
DEPLOY_DIR = MODELS_DIR.parent / "deployment"
DEPLOY_DIR.mkdir(exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Deployment output: {DEPLOY_DIR}")

## 1. Hugging Face Hub Upload Preparation

In [ ]:
try:
    from huggingface_hub import HfApi, login
    HF_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'huggingface_hub', '-q'])
    from huggingface_hub import HfApi, login
    HF_AVAILABLE = True

class HuggingFaceDeployer:
    """
    Deploy models and artifacts to Hugging Face Hub.
    """
    
    def __init__(self, token: str = None):
        self.token = token or CONFIG.get('hf_token') or os.environ.get('HF_TOKEN')
        self.api = HfApi(token=self.token) if self.token else None
        self.repo_id = CONFIG.get('hf_repo', 'Che237/cyberforge-models')
    
    def is_authenticated(self) -> bool:
        """Check if authenticated to Hugging Face"""
        if not self.api:
            return False
        try:
            self.api.whoami()
            return True
        except:
            return False
    
    def create_model_card(self, models_info: Dict) -> str:
        """Create MODEL_CARD.md for Hugging Face"""
        card = f"""
---
license: mit
tags:
  - cybersecurity
  - threat-detection
  - phishing
  - malware
  - security
language:
  - en
---

# CyberForge AI Models

Production-ready machine learning models for cybersecurity threat detection.

## Models Included

| Model | Task | Accuracy | F1 Score | Inference Time |
|-------|------|----------|----------|----------------|
"""
        
        for name, info in models_info.items():
            card += f"| {name} | {info.get('type', 'classification')} | {info.get('accuracy', 0):.4f} | {info.get('f1_score', 0):.4f} | {info.get('inference_time_ms', 0):.2f}ms |\n"
        
        card += f"""

## Usage

### Python

```python
from inference import CyberForgeInference

inference = CyberForgeInference('./models')
result = inference.predict('phishing_detection', features)
```

### API

```python
import requests

response = requests.post(
    'https://huggingface.co/spaces/Che237/cyberforge/predict',
    json={{'model_name': 'phishing_detection', 'features': features}}
)
```

## Model Details

- **Framework**: scikit-learn
- **Python Version**: 3.11+
- **License**: MIT

## Citation

If you use these models, please cite:

```bibtex
@software{{cyberforge2024,
  title = {{CyberForge AI Security Models}},
  year = {{2024}},
  url = {{https://huggingface.co/Che237/cyberforge-models}}
}}
```
"""
        return card
    
    def prepare_upload_package(self, source_dir: Path, output_dir: Path) -> Path:
        """Prepare package for HF upload"""
        hf_dir = output_dir / "huggingface_upload"
        hf_dir.mkdir(exist_ok=True)
        
        # Copy all files from backend package
        if source_dir.exists():
            for item in source_dir.iterdir():
                if item.is_file():
                    shutil.copy(item, hf_dir / item.name)
                elif item.is_dir():
                    shutil.copytree(item, hf_dir / item.name, dirs_exist_ok=True)
        
        return hf_dir

hf_deployer = HuggingFaceDeployer()
print(f"✓ HuggingFace Deployer initialized")
print(f"   Authenticated: {hf_deployer.is_authenticated()}")

## 2. Create Deployment Package

In [ ]:
# Create HF upload package
print("Creating deployment package...\n")

hf_package_dir = hf_deployer.prepare_upload_package(BACKEND_DIR, DEPLOY_DIR)
print(f"✓ Package prepared at: {hf_package_dir}")

# Load models info
manifest_path = BACKEND_DIR / "manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    models_info = manifest.get('models', {})
else:
    models_info = {}

# Create model card
model_card = hf_deployer.create_model_card(models_info)
model_card_path = hf_package_dir / "README.md"
with open(model_card_path, 'w') as f:
    f.write(model_card)

print(f"✓ Model card saved")

## 3. Docker Configuration

In [ ]:
# Generate Dockerfile for ML services
dockerfile_content = '''
# CyberForge ML Services Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy models and code
COPY . .

# Expose port
EXPOSE 8001

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \
    CMD curl -f http://localhost:8001/health || exit 1

# Run server
CMD ["uvicorn", "inference:app", "--host", "0.0.0.0", "--port", "8001"]
'''

dockerfile_path = DEPLOY_DIR / "Dockerfile"
with open(dockerfile_path, 'w') as f:
    f.write(dockerfile_content)

print(f"✓ Dockerfile saved to: {dockerfile_path}")

In [ ]:
# Generate requirements.txt for deployment
requirements_content = '''
# CyberForge ML Requirements
fastapi>=0.104.0
uvicorn>=0.24.0
pydantic>=2.0.0
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
joblib>=1.3.0
python-multipart>=0.0.6
huggingface-hub>=0.19.0
google-genai>=1.0.0
requests>=2.31.0
pyarrow>=14.0.0
'''

requirements_path = DEPLOY_DIR / "requirements.txt"
with open(requirements_path, 'w') as f:
    f.write(requirements_content)

print(f"✓ requirements.txt saved")


In [ ]:
# Generate docker-compose.yml
docker_compose_content = '''
version: "3.8"

services:
  ml-services:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8001:8001"
    environment:
      - GEMINI_API_KEY=${GEMINI_API_KEY}
      - HF_TOKEN=${HF_TOKEN}
    volumes:
      - ./models:/app/models:ro
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8001/health"]
      interval: 30s
      timeout: 10s
      retries: 3
'''

compose_path = DEPLOY_DIR / "docker-compose.yml"
with open(compose_path, 'w') as f:
    f.write(docker_compose_content)

print(f"✓ docker-compose.yml saved")

## 4. Environment Configuration

In [ ]:
# Generate .env.example
env_example = '''
# CyberForge ML Services Environment Configuration

# Hugging Face
HF_TOKEN=your_huggingface_token_here
HF_REPO=Che237/cyberforge-models

# Gemini API
GEMINI_API_KEY=your_gemini_api_key_here

# WebScraper API
WEBSCRAPER_API_KEY=your_webscraper_api_key_here
WEBSCRAPER_API_URL=http://webscrapper.live/api/scrape

# Server Configuration
ML_SERVICE_PORT=8001
ML_SERVICE_HOST=0.0.0.0

# Model Configuration
MODELS_DIR=./models
MAX_INFERENCE_TIME_MS=100
CONFIDENCE_THRESHOLD=0.7
'''

env_path = DEPLOY_DIR / ".env.example"
with open(env_path, 'w') as f:
    f.write(env_example)

print(f"✓ .env.example saved")

## 5. Upload Script

In [ ]:
# Generate HuggingFace upload script
upload_script = '''
#!/usr/bin/env python3
"""
CyberForge - Upload to Hugging Face Hub
"""

import os
import sys
from pathlib import Path
from huggingface_hub import HfApi, login

def main():
    # Get token
    token = os.environ.get('HF_TOKEN')
    if not token:
        print("Error: HF_TOKEN environment variable not set")
        sys.exit(1)
    
    # Login
    login(token=token)
    api = HfApi()
    
    # Configuration
    repo_id = os.environ.get('HF_REPO', 'Che237/cyberforge-models')
    upload_dir = Path('./huggingface_upload')
    
    if not upload_dir.exists():
        print(f"Error: Upload directory not found: {upload_dir}")
        sys.exit(1)
    
    print(f"Uploading to: {repo_id}")
    
    # Upload
    try:
        api.upload_folder(
            folder_path=str(upload_dir),
            repo_id=repo_id,
            repo_type="model",
            commit_message="Update CyberForge ML models"
        )
        print(f"✓ Upload complete: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"Error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()
'''

upload_script_path = DEPLOY_DIR / "upload_to_hf.py"
with open(upload_script_path, 'w') as f:
    f.write(upload_script)

os.chmod(upload_script_path, 0o755)
print(f"✓ Upload script saved")

## 6. Deployment Documentation

In [ ]:
# Generate deployment guide
deployment_guide = f'''
# CyberForge ML Deployment Guide

## Quick Start

### 1. Local Deployment

```bash
# Install dependencies
pip install -r requirements.txt

# Start server
uvicorn inference:app --host 0.0.0.0 --port 8001
```

### 2. Docker Deployment

```bash
# Build and run
docker-compose up -d

# Check logs
docker-compose logs -f
```

### 3. Hugging Face Deployment

```bash
# Set token
export HF_TOKEN=your_token_here

# Upload
python upload_to_hf.py
```

## API Endpoints

### Prediction

```
POST /predict
Content-Type: application/json

{{
    "model_name": "phishing_detection",
    "features": {{
        "url_length": 50,
        "has_https": true,
        ...
    }}
}}
```

### List Models

```
GET /models
```

### Health Check

```
GET /health
```

## Environment Variables

| Variable | Description | Required |
|----------|-------------|----------|
| HF_TOKEN | Hugging Face API token | For HF upload |
| GEMINI_API_KEY | Gemini API key | For AI reasoning |
| WEBSCRAPER_API_KEY | WebScraper API key | For data collection |

## Monitoring

- Health endpoint: `/health`
- Metrics endpoint: `/metrics`
- Logs: `docker-compose logs -f ml-services`

## Troubleshooting

### Model not found
- Check model files exist in `models/` directory
- Verify manifest.json includes the model

### Slow inference
- Check model size
- Consider using lighter model variant
- Verify no resource contention

### API errors
- Check logs for stack traces
- Verify input format matches expected schema
- Ensure all dependencies installed

## Support

For issues and feature requests, please open an issue on GitHub.
'''

guide_path = DEPLOY_DIR / "DEPLOYMENT.md"
with open(guide_path, 'w') as f:
    f.write(deployment_guide)

print(f"✓ Deployment guide saved")

## 7. Final Package Verification

In [ ]:
# Verify deployment package
required_files = [
    'Dockerfile',
    'docker-compose.yml',
    'requirements.txt',
    '.env.example',
    'upload_to_hf.py',
    'DEPLOYMENT.md',
    'huggingface_upload/README.md'
]

print("Verifying deployment package...\n")

all_present = True
for file in required_files:
    path = DEPLOY_DIR / file
    exists = path.exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {file}")
    if not exists:
        all_present = False

print(f"\n{'✓ All files present' if all_present else '✗ Some files missing'}")

## 8. Summary

In [ ]:
# Calculate package stats
total_files = len(list(DEPLOY_DIR.rglob('*')))
total_size = sum(f.stat().st_size for f in DEPLOY_DIR.rglob('*') if f.is_file())

print("\n" + "=" * 60)
print("DEPLOYMENT ARTIFACTS COMPLETE")
print("=" * 60)

print(f"""
🚀 Deployment Package Ready:
   - Location: {DEPLOY_DIR}
   - Files: {total_files}
   - Total size: {total_size / (1024*1024):.2f} MB

📦 Package Contents:
   - Dockerfile: Container configuration
   - docker-compose.yml: Multi-service orchestration
   - requirements.txt: Python dependencies
   - .env.example: Environment template
   - upload_to_hf.py: HuggingFace upload script
   - DEPLOYMENT.md: Deployment guide
   - huggingface_upload/: HF Hub package

🔧 Deployment Options:
   1. Local: uvicorn inference:app --port 8001
   2. Docker: docker-compose up -d
   3. HuggingFace: python upload_to_hf.py

📋 Next Steps:
   1. Set environment variables (copy .env.example to .env)
   2. Choose deployment method
   3. Verify health endpoint
   4. Integrate with backend services

🎉 CyberForge ML Pipeline Complete!
""")
print("=" * 60)

## 9. Optional: Upload to Hugging Face

In [ ]:
# Upload trained models and artifacts to Hugging Face Hub
# Force authentication with explicit token from config
from huggingface_hub import HfApi
import os

hf_token = CONFIG.get('hf_token') or os.environ.get('HF_TOKEN')
repo_id = CONFIG.get('hf_repo', 'Che237/cyberforge-models')

if hf_token:
    print(f"Uploading to Hugging Face Hub: {repo_id}")
    try:
        api = HfApi(token=hf_token)
        
        # Create repo if it doesn't exist
        try:
            api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
            print(f"✓ Repository ready: {repo_id}")
        except Exception as e:
            print(f"  Repo exists or error: {e}")
        
        # Upload the package
        api.upload_folder(
            folder_path=str(hf_package_dir),
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload CyberForge ML models and deployment artifacts"
        )
        print(f"✓ Successfully uploaded to: https://huggingface.co/{repo_id}")
        print(f"✓ Models are now accessible via HuggingFace!")
    except Exception as e:
        print(f"✗ Upload error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("✗ HF_TOKEN not found!")
    print("  Set hf_token in notebook_config.json or HF_TOKEN environment variable")